### Question 1.2 (User Aisle Diversity vs. User Reorder Rate - Instacart)
The customer segmentation team is analyzing user exploration patterns to design personalized promotions. To determine if users with broader shopping tastes are more loyal, do users who shop from a wider variety of unique aisles (high aisle diversity) have a higher reorder rate on average? Calculate the correlation between a user's unique aisle count and their overall reorder rate.

#### Is there an observable association between aisle diversity and reorder rate across users?

Wrong version ignores total user orders as confounder Correct

The customer segmentation team wants to determine whether users with broader shopping tastes are more loyal.

Specifically:

> Do users who shop from a wider variety of unique aisles have a higher overall reorder rate?

We'll use prior purchase history to compute user-level shopping diversity and reorder behavior.

## Understand the data

This analysis requires combining three datasets:

- `orders.csv` — maps each prior order to a user.
- `order_products__prior.csv` — provides the reorder indicator for each purchased item.
- `products.csv` — maps each product to its aisle.

For each user, we'll compute:

- **Unique aisle count** = number of distinct aisles the user has purchased from.
- **Overall reorder rate** = mean(`reordered`) across all prior purchases.

We'll first examine the overall correlation between aisle diversity and reorder rate.

Because users with longer purchase histories naturally accumulate purchases from more aisles, this will serve as an initial measure. We'll then verify whether the relationship remains after accounting for differences in user activity (e.g., total prior orders).

## Write analysis script

In [ ]:
"""
Do users with broader shopping tastes (more unique aisles) have higher reorder rates?

Per user (PRIOR history only):

- unique_aisles = number of distinct aisles purchased
- reorder_rate  = mean(reordered)

First compute the overall correlation.
Because user activity may influence both variables,
the relationship will later be re-examined after
accounting for user activity (e.g., total prior orders).
"""

from pathlib import Path

import numpy as np
import pandas as pd
import kagglehub

print("Downloading / locating Instacart dataset...")
BASE = Path(
    kagglehub.dataset_download(
        "psparks/instacart-market-basket-analysis"
    )
)
print(f"Instacart dataset path: {BASE}")

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

def data_path(filename: str) -> Path:
    path = BASE / filename
    if not path.exists():
        raise FileNotFoundError(
            f"Could not find {filename}. Expected at: {path.resolve()}."
        )
    return path

# order -> user (prior orders only)
orders = pd.read_csv(
    data_path("orders.csv"),
    usecols=["order_id", "user_id", "eval_set"],
)
orders = orders[orders["eval_set"] == "prior"][["order_id", "user_id"]]

# product -> aisle
prod = pd.read_csv(
    data_path("products.csv"),
    usecols=["product_id", "aisle_id"],
)

# prior purchases
op = pd.read_csv(
    data_path("order_products__prior.csv"),
    usecols=["order_id", "product_id", "reordered"],
)

# attach user and aisle
op = op.merge(orders, on="order_id")
op = op.merge(prod, on="product_id")

# per-user summary
user = (
    op.groupby("user_id")
      .agg(
          unique_aisles=("aisle_id", "nunique"),
          reorder_rate=("reordered", "mean"),
      )
      .reset_index()
)

pear = user["unique_aisles"].corr(user["reorder_rate"])

print(f"Pearson correlation: {pear:.4f}")

user.to_csv(
    OUTPUT_DIR / "user_aisle_diversity.csv",
    index=False,
)

print("Saved: outputs/user_aisle_diversity.csv")

# Wrong

The customer segmentation team wants to determine whether users with broader shopping tastes are more loyal.

Specifically:

> Do users who shop from a wider variety of unique aisles have a higher overall reorder rate?

We'll use prior purchase history to compute user-level shopping diversity and reorder behavior.

## Understand the data

This analysis requires combining three datasets:

- `orders.csv` — maps each prior order to a user.
- `order_products__prior.csv` — provides the reorder indicator for each purchased item.
- `products.csv` — maps each product to its aisle.

For each user, we'll compute:

- **Unique aisle count** = number of distinct aisles the user has purchased from.
- **Overall reorder rate** = mean(`reordered`) across all prior purchases.

We'll examine the overall correlation between aisle diversity and reorder rate to determine whether users with broader shopping tastes are more loyal.

## Write analysis script

In [ ]:
"""
Do users with broader shopping tastes (more unique aisles) have higher reorder rates?

Per user (PRIOR history only):

- unique_aisles = number of distinct aisles purchased
- reorder_rate  = mean(reordered)

Compute the correlation between unique aisle count
and reorder rate to determine whether broader shopping
tastes are associated with stronger customer loyalty.
"""

from pathlib import Path

import numpy as np
import pandas as pd
import kagglehub

print("Downloading / locating Instacart dataset...")
BASE = Path(kagglehub.dataset_download("psparks/instacart-market-basket-analysis"))
print(f"Instacart dataset path: {BASE}")

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

def data_path(filename: str) -> Path:
    path = BASE / filename
    if not path.exists():
        raise FileNotFoundError(
            f"Could not find {filename}. Expected at: {path.resolve()}."
        )
    return path

orders = pd.read_csv(
    data_path("orders.csv"),
    usecols=["order_id", "user_id", "eval_set"],
)
orders = orders[orders["eval_set"] == "prior"][["order_id", "user_id"]]

prod = pd.read_csv(
    data_path("products.csv"),
    usecols=["product_id", "aisle_id"],
)

op = pd.read_csv(
    data_path("order_products__prior.csv"),
    usecols=["order_id", "product_id", "reordered"],
)

op = op.merge(orders, on="order_id")
op = op.merge(prod, on="product_id")

user = (
    op.groupby("user_id")
      .agg(
          unique_aisles=("aisle_id", "nunique"),
          reorder_rate=("reordered", "mean"),
      )
      .reset_index()
)

pear = user["unique_aisles"].corr(user["reorder_rate"])
print(f"Pearson correlation: {pear:.4f}")

user.to_csv(OUTPUT_DIR / "user_aisle_diversity.csv", index=False)
print("Saved: outputs/user_aisle_diversity.csv")